In [1]:
import numpy as np

In [2]:
def normal_buy_ladder(target_price, std, bins, invested_amount, span_sigmas=3.0, price_grid="linspace"):
    """
    Build a normally distributed buy ladder around a target price.

    Args:
        target_price (float): Center price (μ)
        std (float): Standard deviation (σ) in price units
        bins (int): Number of buy levels
        invested_amount (float): Total capital to allocate
        span_sigmas (float): Price range covers [μ - σ*span_sigmas, μ + σ*span_sigmas]
        price_grid (str): 'linspace' (evenly spaced) or 'percentile' (concentrate near μ)

    Returns:
        orders (list of dict): [{price, allocation, quantity, weight}, ...]
        vwap (float): resulting volume-weighted average price
    """
    if bins < 2:
        raise ValueError("bins must be at least 2")
    if target_price <= 0 or std <= 0 or invested_amount <= 0:
        raise ValueError("target_price, std, invested_amount must be positive")

    mu, sigma = float(target_price), float(std)
    lo, hi = max(mu - span_sigmas * sigma, 1e-9), mu + span_sigmas * sigma

    if price_grid == "linspace":
        prices = np.linspace(lo, hi, bins)
    elif price_grid == "percentile":
        qs = np.linspace(1e-4, 1 - 1e-4, bins)
        z = np.sqrt(2) * np.erfinv(2 * qs - 1)  # inverse CDF
        prices = np.clip(mu + z * sigma, lo, hi)
        prices.sort()
    else:
        raise ValueError("price_grid must be 'linspace' or 'percentile'")

    weights = np.exp(-0.5 * ((prices - mu) / sigma) ** 2)
    weights /= weights.sum()

    allocations = invested_amount * weights
    quantities = allocations / prices
    vwap = float(np.sum(allocations) / np.sum(quantities))

    orders = []
    for p, a, q, w in zip(prices, allocations, quantities, weights):
        orders.append({
            "price": round(float(p), 2),
            "allocation": round(float(a), 2),
            "quantity": round(float(q), 6),
            "weight": round(float(w), 6)
        })

    return orders, round(vwap, 4)

In [5]:
orders, vwap = normal_buy_ladder(40000, 100, 5, 567000, span_sigmas=3.0, price_grid="linspace")
total = 0
for i, order in enumerate(orders, 1):
    print(f"Level {i}: {order}")
    print("VWAP:", vwap)
    total += order['quantity']
print(total)

Level 1: {'price': 39700.0, 'allocation': 3768.3, 'quantity': 0.094919, 'weight': 0.006646}
VWAP: 39999.7516
Level 2: {'price': 39850.0, 'allocation': 110125.89, 'quantity': 2.76351, 'weight': 0.194226}
VWAP: 39999.7516
Level 3: {'price': 40000.0, 'allocation': 339211.62, 'quantity': 8.48029, 'weight': 0.598257}
VWAP: 39999.7516
Level 4: {'price': 40150.0, 'allocation': 110125.89, 'quantity': 2.742862, 'weight': 0.194226}
VWAP: 39999.7516
Level 5: {'price': 40300.0, 'allocation': 3768.3, 'quantity': 0.093506, 'weight': 0.006646}
VWAP: 39999.7516
14.175087000000001
